# Linguistic Diagnostics

In [1]:
import json
from collections import defaultdict
from pathlib import Path

from edu_eval.linguistic.analyzer import analyze_text

In [2]:
PROJECT_ROOT = Path.cwd().parent
INPUT_PATH = PROJECT_ROOT / "results" / "generation" / "pilot" / "smoke_qwen" / "controlled_outputs.jsonl"
OUT_DIR = PROJECT_ROOT / "results" / "linguistic"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with INPUT_PATH.open("r", encoding="utf-8") as f:
    outputs = [json.loads(line) for line in f if line.strip()]

In [3]:
by_grade: dict[str, list[dict]] = defaultdict(list)
features_path = OUT_DIR / "pilot_smoke_features.jsonl"
with features_path.open("w", encoding="utf-8") as out:
    for r in outputs:
        features = analyze_text(r["output_text"])
        by_grade[r["target_grade"]].append(features)
        out.write(json.dumps({"model_id": r["model_id"], "case_id": r["case_id"], "target_grade": r["target_grade"], "features": features}, ensure_ascii=False) + "\n")
print(f"{sum(len(v) for v in by_grade.values())} records")
for grade in sorted(by_grade):
    feats = by_grade[grade]
    mwps = sum(f["words_per_sentence"] for f in feats) / len(feats)
    mttr = sum(f["lexical_diversity_ttr"] for f in feats) / len(feats)
    print(f"{grade}: n={len(feats)} wps={mwps:.1f} ttr={mttr:.3f}")

40 records
SD6: n=10 wps=10.6 ttr=0.561
SMA10: n=10 wps=15.8 ttr=0.612
SMP7: n=10 wps=11.8 ttr=0.587
SMP9: n=10 wps=12.5 ttr=0.609
